# RAG Agent

上一节我们完成了知识库的构建（文档加载→切分→向量化→存储），本节将聚焦于如何构建RAG Agent，将检索与生成结合。

本章会分为两大部分：
- RAG的常见架构
- RAG知识检索的优化策略

## 1. 准备知识库
在正式开始前，我们先利用上节课知识，准备好一个知识库，方便后续测试知识检索：

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import DashScopeEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

# 读取markdown数据
docs = ""
with open("./resources/中二知识笔记.md", "r", encoding="utf-8") as f:
    lines = f.readlines()
    docs = "".join(line for line in lines)

# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

# 切分文档
chunks = markdown_splitter.split_text(docs)


idx = 1
# 定义方法，用来给文档内容拼上三级标题，让文档不丢失其所属章节，顺便加个id
def handle_doc_header(doc: Document):
    global idx
    m = doc.metadata
    doc.page_content = f"### {m['Header 3']}\n{doc.page_content}"
    doc.id = f"doc_{idx}"
    idx = idx + 1
    return doc
ds = [handle_doc_header(doc) for doc in chunks]

# 创建向量库和检索器
embeddings = DashScopeEmbeddings(
    model="text-embedding-v3", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)
vectorstore = InMemoryVectorStore(embeddings)

# 添加文档
vectorstore.add_documents(ds)

print(f"知识库已就绪，{len(chunks)} 个文档")